# 106. 类别不平衡与Top-K Lift

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 21 / 34 步：从模型分数走向业务评价与阈值**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** ROC、PR曲线与决策阈值  →  **本章任务：** 类别不平衡与Top-K Lift  →  **下一步：** 概率校准与Brier Score
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

做数据分析时，我们常遇到「想找的目标在数据里只占很小一部分」的情况，比如银行营销数据里真正会购买的人往往不到一成。



## 本章目标

学完本章，你将能够：

- **理解**：理解「类别不平衡与Top-K Lift」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「类别不平衡与Top-K Lift」的关键输出指标。
- **迁移**：能把「类别不平衡与Top-K Lift」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 106.1 核心概念

**背景引入**：做数据分析时，我们常遇到「想找的目标在数据里只占很小一部分」的情况，比如银行营销数据里真正会购买的人往往不到一成。这时候只报准确率很容易被「绝大多数都是不购买的人」骗过去，真正该问的是：把模型评分最高的一批人拎出来，里面命中的比例是不是比随机抓一批人明显更高。下面就用几个指标把「类别不平衡」和「Top-K Lift」算清楚，帮你判断模型到底有没有用。

- PR 基线约等于正类比例
- Lift@k=Top-k正类率/总体正类率（打个比方：随机捞一批人，命中率只是“基线”；若模型能把最可能成交的排最前，头几名命中率明显更高，才说明有真本事——否则跟瞎抓一样。）
- 类别权重改变损失贡献
- 响应概率不等于干预增量


## 106.2 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 数据与问题定义 | `pd.read_csv()`、`.astype()`、`df['target']`、`df[features]` | 先明确样本、特征、目标和验证方式，再训练模型。 | 只报告准确率 |
| 模型、公式与诊断 | `model.predict_proba()`、`pd.DataFrame()`、`y_test.to_numpy()`、`ranked.head()` | 把核心数学量映射到 sklearn 输出，并检查泛化表现。 | 把过采样放在切分之前 |


## 106.3 示例 1：数据与问题定义

先明确样本、特征、目标和验证方式，再训练模型。


<!-- math-foundation:chapter-106 -->
### 数学推导｜Top-K Lift 衡量名单浓度

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜总体正类率给出随机基线。** $p_{base}=P/N$。

**第 2 步｜分数排序后取前 $K$。** Top-K 命中率 $p_K=TP_K/K$，正类覆盖率 $Recall@K=TP_K/P$。

**第 3 步｜比较浓度。** $Lift@K=p_K/p_{base}$。累计增益还可写成

$$
Gain@K=\frac{TP_K}{P}=Recall@K
$$

Lift 侧重名单“纯度提升”，Gain 侧重找回了多少全部正类，两者应一起报告。

**把上面的关系收束为本章计算式：**

$$
Lift@K=\frac{TP_K/K}{P/N}
$$

**符号解释：** $TP_K/K$ 是 Top-K 名单命中率，$P/N$ 是总体正类率。

**代码对应：** 概率降序后计算多个 K 或预算比例下的 lift、命中数和覆盖率。

**使用边界：** Lift 依赖评估样本的基准率；不同时间或人群间不能脱离基准直接比较。


In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

df = pd.read_csv("/datasets/bank_marketing_full.csv", sep=";")
df["target"] = (df.y == "yes").astype(int)
features = [
    "age",
    "campaign",
    "pdays",
    "previous",
    "emp.var.rate",
    "cons.price.idx",
    "euribor3m",
    "nr.employed",
]
X_train, X_test, y_train, y_test = train_test_split(
    df[features], df.target, stratify=df.target, random_state=95
)


**练一练**：示例 1 已经把数据读进来，并用 `df["target"]` 把「是否答复」编码成 0/1，还做了分层切分。在继续训练之前，先看清这个问题有多「不平衡」——它决定了后面该用 PR-AUC / Top-K Lift 而不是只盯准确率。请把下面 `n_pos`、`n_neg` 两个空补上，分别统计 `df` 中正类(1)与负类(0)的样本数，运行自检。提示：`df["target"].sum()` 就是正类个数，`(df["target"] == 0).sum()` 就是负类个数，两者相加应等于 `len(df)`。


In [ ]:
# 请在下方填写代码
# 目标：统计 df 中正类(1)与负类(0)的样本数量，判断这个分类问题是否类别不平衡。
# 提示：df["target"].sum() 统计正类个数；(df["target"] == 0).sum() 统计负类个数。

# --- 你的代码 ---
n_pos = None  # 请填：统计正类(1)个数的代码
n_neg = None  # 请填：统计负类(0)个数的代码
# --- 你的代码结束 ---


In [ ]:
# 完整答案
n_pos = int(df["target"].sum())  # 正类(1)的数量
n_neg = int((df["target"] == 0).sum())  # 负类(0)的数量


## 106.4 示例 2：模型、公式与诊断

把核心数学量映射到 sklearn 输出，并检查泛化表现。


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

model = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=500, class_weight="balanced")
).fit(X_train, y_train)
prob = model.predict_proba(X_test)[:, 1]
ranked = pd.DataFrame({"y": y_test.to_numpy(), "p": prob}).sort_values(
    "p", ascending=False
)
for share in [0.05, 0.1, 0.2]:
    top = ranked.head(int(len(ranked) * share))
    print(
        share,
        "PR-AUC",
        round(average_precision_score(y_test, prob), 3),
        "lift",
        round(top.y.mean() / ranked.y.mean(), 2),
        "coverage",
        round(top.y.sum() / ranked.y.sum(), 3),
    )


## 106.5 建模流程提醒

1. **定义问题**：写清楚样本粒度、预测时点、目标变量和业务代价。
2. **建立基线**：先用均值、规则或 Dummy 模型得到最低可接受结果。
3. **准备数据**：只用预测时点可获得的信息，避免目标泄漏和时间穿越。
4. **训练与验证**：在训练/验证数据上选择方案，测试集只用于最终估计泛化表现。
5. **评价与解释**：同时看总体指标、错误切片和结果边界，不能只报一个分数。


## 106.6 独立迁移练习

在不改变数据切分和指标的前提下，比较基线与一个模型设置。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# TODO: 在此粘贴或改写最接近的示例。
# 记录：我改了什么？预期会发生什么？实际观察到什么？
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print({"修改": change_note, "预期": expected_change, "观察": observed_change})


## 106.7 本章实训：模型与基线比较

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X = pd.DataFrame(
    {"visits": [1, 2, 3, 4, 5, 6], "discount": [0, 0, 1, 1, 1, 2]}
)
y = np.array([12, 15, 19, 23, 27, 31])
baseline = DummyRegressor(strategy="mean").fit(X, y)
model = LinearRegression().fit(X, y)
print("基线预测：", np.round(baseline.predict(X[:2]), 2))
print("模型预测：", np.round(model.predict(X[:2]), 2))
print("基线MAE：", round(mean_absolute_error(y, baseline.predict(X)), 2))
print("模型MAE：", round(mean_absolute_error(y, model.predict(X)), 2))


### 106.7.1 第一个结果怎么读

复杂模型之前先建立基线。只有在同一数据切分和同一指标下超过基线，模型才值得继续分析。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
X_changed = X.copy()
X_changed["visits"] = X_changed["visits"] + 1
changed_prediction = model.predict(X_changed)
print("原始前2个预测：", np.round(model.predict(X[:2]), 2))
print("访问次数+1后的预测：", np.round(changed_prediction[:2], 2))
print("预测变化：", np.round(changed_prediction[:2] - model.predict(X[:2]), 2))


### 106.7.2 第二个结果怎么读

只把一个特征整体加 1，观察预测变化。这个实验只能说明模型的预测响应，不能直接证明真实世界的因果关系。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 106.8 错误恢复：模型特征泄漏怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

data = pd.DataFrame(
    {
        "visits": [2, 4, 6],
        "duration_after_call": [30, 80, 120],
        "target": [0, 1, 1],
    }
)
forbidden = {"target", "duration_after_call"}
features = [column for column in data.columns if column not in forbidden]
print("禁止使用：", sorted(forbidden))
print("安全特征：", features)
print("原因：特征必须在预测时点已经可获得。")


### 106.8.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

如果一个字段在结果发生之后才产生，它即使与目标高度相关，也不能作为预测特征。先定义预测时点，再列可用字段。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 106.9 易错点提醒

- 只报告准确率
- 把过采样放在切分之前
- Top-K 结果不报告名单规模
- 把高响应评分称为因果 uplift


## 106.10 练习与作业

1. 修改一个关键参数并重新运行
2. 记录指标变化并解释原因
3. 检查结论是否依赖测试集或隐藏泄漏

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 106.11 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“修改一个关键参数并重新运行”。
2. **独立完成**：不复制示例代码，完成“记录指标变化并解释原因”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“检查结论是否依赖测试集或隐藏泄漏”，用一两句话说明你修改了什么。

### 106.11.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 106.11.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
k = int(len(ranked) * 0.1)
top10 = ranked.head(k)
practice_lift = top10.y.mean() / ranked.y.mean()
print("Top10 lift:", round(practice_lift, 2))


## 106.12 小结

针对不平衡分类比较类别权重、PR-AUC、Top-K Lift 和覆盖率。


### 106.12.1 你已经掌握

- 建立 Dummy 基线
- 使用 class_weight
- 计算 Top-K Lift
- 结合营销或审核容量评价模型


### 106.12.2 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。


### 106.12.3 需要注意

- 只报告准确率
- 把过采样放在切分之前
- Top-K 结果不报告名单规模
- 把高响应评分称为因果 uplift


### 106.12.4 完成检查

- [ ] 能够建立 Dummy 基线
- [ ] 能够使用 class_weight
- [ ] 能够计算 Top-K Lift
- [ ] 能够结合营销或审核容量评价模型


### 106.12.5 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。
